In [1]:
# Import packages
import numpy as np
import pandas as pd
import joblib

# Import libraries for scaling and transforming features
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Import the model for KNearestNeighborClassifier
from sklearn.neighbors import KNeighborsClassifier

# Import the model for DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier

# Import the model for Support Vector Machine (SVM) by using LinearSVC
from sklearn.svm import LinearSVC

# Import the model for XGBoost Classifier
from xgboost.sklearn import XGBClassifier

# Import the model for RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, accuracy_score

In [2]:
## Interpret the dataset files

# Load dataset files
training_data = pd.read_csv('UNSW_NB15_training-set.csv')
testing_data = pd.read_csv('UNSW_NB15_testing-set.csv')

# Remove all invalid records
training_data = training_data.dropna()
testing_data = testing_data.dropna()

# Remove the 'id' feature because it just indicates the row index of records
training_data.drop(columns=['id'], inplace=True)
testing_data.drop(columns=['id'], inplace=True)

# Initially comprehend the dataset
# Total number of rows => Network traffic records
# Total number of columns => Attributes, Features, and Labels for each record
# print('Training shape:', training_data.shape)

# Preview the training dataset
training_data.head()

# Preview the attributes
# training_data.info()

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.121478,tcp,-,FIN,6,4,258,172,74.087490,252,...,1,1,0,0,0,1,1,0,Normal,0
1,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,62,...,1,2,0,0,0,1,6,0,Normal,0
2,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,62,...,1,3,0,0,0,2,6,0,Normal,0
3,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,62,...,1,3,1,1,0,2,1,0,Normal,0
4,0.449454,tcp,-,FIN,10,6,534,268,33.373826,254,...,1,40,0,0,0,2,39,0,Normal,0


In [3]:
# Define features and target
# Features
X_training = training_data.drop(columns=['label', 'attack_cat'])
# Target
y_training = training_data['attack_cat']

# Features
X_testing = testing_data.drop(columns=['label', 'attack_cat'])
# Target
y_testing = testing_data['attack_cat']


In [4]:
# Initialize label_encoder
label_encoder = LabelEncoder()

# Convert the datatype of Target from string to number
y_training_encoded = label_encoder.fit_transform(y_training)
y_testing_encoded = label_encoder.transform(y_testing)

# List all categories of attacks
print(label_encoder.classes_)
# Result after converting
print(y_training_encoded)

['Analysis' 'Backdoor' 'DoS' 'Exploits' 'Fuzzers' 'Generic' 'Normal'
 'Reconnaissance' 'Shellcode' 'Worms']
[6 6 6 ... 5 5 5]


In [5]:
# Identify all features whose datatype is string
string_features = X_training.select_dtypes(include=['object']).columns
print(string_features)

# Identify all features whose datatype is numerical
numerical_features = X_training.select_dtypes(include=['int64', 'float64']).columns
print(numerical_features)

Index(['proto', 'service', 'state'], dtype='object')
Index(['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl',
       'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit',
       'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean',
       'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src',
       'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
       'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd',
       'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports'],
      dtype='object')


In [6]:
# Transform identified features
# Initialize numerical_scale
numerical_scale = StandardScaler()
# Initialize string_scale
string_scale = OneHotEncoder(handle_unknown='ignore')

# Create the preprocessor
preprocessor = ColumnTransformer(
    transformers=[('numerical', numerical_scale, numerical_features), ('string', string_scale, string_features)]
)

In [7]:
# Start preprocessing the dataset
X_training_processed = preprocessor.fit_transform(X_training)
X_testing_processed = preprocessor.transform(X_testing)

# Result
print(X_training_processed)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7364322 stored elements and shape (175341, 194)>
  Coords	Values
  (0, 0)	-0.19102880916047862
  (0, 1)	-0.10445580785769364
  (0, 2)	-0.13576880257230514
  (0, 3)	-0.04913361751324256
  (0, 4)	-0.10272556475986418
  (0, 5)	-0.5763712666832832
  (0, 6)	0.7038391473682516
  (0, 7)	1.578100452557122
  (0, 8)	-0.3898974255721421
  (0, 9)	-0.27369954484801623
  (0, 10)	-0.07503991948315326
  (0, 11)	-0.1317586669446255
  (0, 12)	-0.1327880918604115
  (0, 13)	-0.08088549744729005
  (0, 14)	-0.10999661223605284
  (0, 15)	-0.14590460407244804
  (0, 16)	1.0924562065765335
  (0, 17)	-0.2563918593487938
  (0, 18)	0.9111229950269237
  (0, 19)	1.1032437785968037
  (0, 20)	-0.5216596150973576
  (0, 21)	-0.484345974086776
  (0, 22)	-0.5030137004862794
  (0, 23)	-0.4580479098424511
  (0, 24)	-0.31424023983176297
  :	:
  (175340, 17)	-0.7151765320679372
  (175340, 18)	-0.7155687700341328
  (175340, 19)	-0.9064315437341032
  (175340, 20)	-0.

In [8]:
## K-NearestNeighbor (K-NN) Classification Model
# Initialize a K-NN classifier model
knn_clf = KNeighborsClassifier(n_neighbors=3)

# Train the K-NN Model
knn_clf.fit(X_training_processed, y_training_encoded)

# Make predictions on the testing data
y_prediction = knn_clf.predict(X_testing_processed)

# Accuracy
accuracy = accuracy_score(y_testing_encoded, y_prediction)
# Classification Report
clf_report = classification_report(y_testing_encoded, y_prediction, target_names=label_encoder.classes_)

# Evaluate the results
print("Accuracy:", accuracy)
print("\nClassification Report:\n", clf_report)

Accuracy: 0.6996429091969101

Classification Report:
                 precision    recall  f1-score   support

      Analysis       0.03      0.13      0.05       677
      Backdoor       0.03      0.14      0.05       583
           DoS       0.21      0.15      0.17      4089
      Exploits       0.59      0.64      0.61     11132
       Fuzzers       0.25      0.47      0.33      6062
       Generic       1.00      0.96      0.98     18871
        Normal       0.93      0.71      0.81     37000
Reconnaissance       0.60      0.61      0.61      3496
     Shellcode       0.38      0.25      0.30       378
         Worms       0.71      0.11      0.20        44

      accuracy                           0.70     82332
     macro avg       0.47      0.42      0.41     82332
  weighted avg       0.78      0.70      0.73     82332



In [9]:
## Decision Tree Classification Model
# Initialize a decision tree classifier model
decision_tree_clf = DecisionTreeClassifier(criterion='entropy', max_depth=10)

# Train the decision tree model
decision_tree_clf.fit(X_training_processed, y_training_encoded)

# Make predictions on the testing data
y_prediction = decision_tree_clf.predict(X_testing_processed)

# Accuracy
accuracy = accuracy_score(y_testing_encoded, y_prediction)
# Classification Report
clf_report = classification_report(y_testing_encoded, y_prediction, target_names=label_encoder.classes_)

# Evaluate the results
print("Accuracy:", accuracy)
print("\nClassification Report:\n", clf_report)

Accuracy: 0.6986347957051936

Classification Report:
                 precision    recall  f1-score   support

      Analysis       0.03      0.03      0.03       677
      Backdoor       0.09      0.12      0.10       583
           DoS       0.43      0.06      0.10      4089
      Exploits       0.55      0.86      0.67     11132
       Fuzzers       0.19      0.50      0.28      6062
       Generic       1.00      0.96      0.98     18871
        Normal       0.99      0.63      0.77     37000
Reconnaissance       0.88      0.80      0.84      3496
     Shellcode       0.17      0.76      0.27       378
         Worms       0.30      0.45      0.36        44

      accuracy                           0.70     82332
     macro avg       0.46      0.52      0.44     82332
  weighted avg       0.82      0.70      0.72     82332



In [10]:
## Linear Support Vector Classification Model
# Initialize a LinearSVC
linearSVC_clf = LinearSVC()

# Train the LinearSVC
linearSVC_clf.fit(X_training_processed, y_training_encoded)

# Make predictions on the testing data
y_prediction = linearSVC_clf.predict(X_testing_processed)

# Accuracy
accuracy = accuracy_score(y_testing_encoded, y_prediction)
# Classification Report
clf_report = classification_report(y_testing_encoded, y_prediction, target_names=label_encoder.classes_)

# Evaluate the results
print("Accuracy:", accuracy)
print("\nClassification Report:\n", clf_report)

Accuracy: 0.6840960987222465

Classification Report:
                 precision    recall  f1-score   support

      Analysis       0.00      0.00      0.00       677
      Backdoor       0.00      0.00      0.00       583
           DoS       0.15      0.07      0.10      4089
      Exploits       0.52      0.79      0.63     11132
       Fuzzers       0.24      0.65      0.35      6062
       Generic       0.99      0.96      0.98     18871
        Normal       0.96      0.62      0.75     37000
Reconnaissance       0.48      0.64      0.55      3496
     Shellcode       0.00      0.00      0.00       378
         Worms       0.00      0.00      0.00        44

      accuracy                           0.68     82332
     macro avg       0.33      0.37      0.34     82332
  weighted avg       0.77      0.68      0.70     82332



c:\Users\kimvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\kimvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\kimvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [1]:
## Random Forest Classification Model
# Initialize a random forest classifier model
random_forest_clf = RandomForestClassifier(n_estimators=100, criterion='entropy')

# Train the random forest model
random_forest_clf.fit(X_training_processed, y_training_encoded)

# Make predictions on the testing data
y_prediction = random_forest_clf.predict(X_testing_processed)

# Accuracy
accuracy = accuracy_score(y_testing_encoded, y_prediction)
# Classification Report
clf_report = classification_report(y_testing_encoded, y_prediction, target_names=label_encoder.classes_)

# Evaluate the results
print("Accuracy:", accuracy)
print("\nClassification Report:\n", clf_report)

NameError: name 'RandomForestClassifier' is not defined

In [12]:
## Extreme Gradient Boost (XGBoost) Classification Model
# Initialize a xgboost classifier model
xgboost_clf = XGBClassifier(object='multi:softprob', n_estimators=100)

# Train the xgboost model
xgboost_clf.fit(X_training_processed, y_training_encoded)

# Make predictions on the testing data
y_prediction = xgboost_clf.predict(X_testing_processed)

# Accuracy
accuracy = accuracy_score(y_testing_encoded, y_prediction)
# Classification Report
clf_report = classification_report(y_testing_encoded, y_prediction, target_names=label_encoder.classes_)

# Evaluate the results
print("Accuracy:", accuracy)
print("\nClassification Report:\n", clf_report)

c:\Users\kimvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:30:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "object" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Accuracy: 0.7660690861390468

Classification Report:
                 precision    recall  f1-score   support

      Analysis       0.05      0.10      0.06       677
      Backdoor       0.04      0.10      0.06       583
           DoS       0.38      0.12      0.19      4089
      Exploits       0.61      0.85      0.71     11132
       Fuzzers       0.30      0.56      0.39      6062
       Generic       1.00      0.97      0.98     18871
        Normal       0.96      0.76      0.85     37000
Reconnaissance       0.93      0.81      0.87      3496
     Shellcode       0.34      0.79      0.47       378
         Worms       0.66      0.48      0.55        44

      accuracy                           0.77     82332
     macro avg       0.53      0.55      0.51     82332
  weighted avg       0.83      0.77      0.78     82332



In [13]:
## Summary of Accuracy Score

# K-NN Model: 0.699
# Decision Tree Model: 0.698
# LinearSVC: 0.68
# Random Forest Model: 0.75
# XGBoost Model: 0.766

In [15]:
## Save the preprocessor
joblib.dump(preprocessor, 'preprocessor.pkl')

## Save the XGBoost Model after training
joblib.dump(xgboost_clf, 'xgboost_model.pkl')

## Save the Label Encoder
joblib.dump(label_encoder, 'label_encoder.pkl')

['label_encoder.pkl']